# 1. Experiment Aim

This final experiment asks how closely three approximate machine-unlearning methods reproduce scenario-matched Full Retraining while preserving retained utility and reducing runtime. Every method–scenario run begins independently from the same verified local experimental Qwen baseline. The historical A100 baseline remains immutable and is used only for transparent cross-environment reproduction evidence.

# 2. Setup and Device

Execution is MPS-first, with CUDA second and CPU fallback. Standard Transformers and PEFT replace the original Unsloth runtime. Install a PEFT version compatible with the saved adapter before running, for example `python -m pip install "peft>=0.20"`.

In [41]:
from pathlib import Path
import gc, hashlib, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from IPython.display import Markdown, display
from scipy.stats import ks_2samp
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score)
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration
try:
    from peft import PeftModel
except ImportError as error:
    raise ImportError('PEFT is required. Install with: python -m pip install "peft>=0.20"') from error

if torch.backends.mps.is_available(): DEVICE = torch.device('mps')
elif torch.cuda.is_available(): DEVICE = torch.device('cuda')
else: DEVICE = torch.device('cpu')
MODEL_DTYPE = torch.float16 if DEVICE.type in {'mps', 'cuda'} else torch.float32
SEED, THRESHOLD, RUN_FINAL_EXPERIMENT = 42, 0.55, True
print('Selected device:', DEVICE, '| model dtype:', MODEL_DTYPE)

Selected device: mps | model dtype: torch.float16


In [42]:
def locate_final_submission():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        direct = candidate if candidate.name == 'final_submission' else candidate / 'code' / 'final_submission'
        if (direct / 'data' / 'final' / 'kidney_transplant_assessments.csv').is_file(): return direct.resolve()
    raise FileNotFoundError('Could not locate code/final_submission.')

FINAL = locate_final_submission()
MODEL_ROOT, RESULT_ROOT = FINAL / 'models' / 'qwen', FINAL / 'results' / 'qwen'
BASELINE_MODEL = MODEL_ROOT / 'baseline'
BASELINE_ADAPTER = BASELINE_MODEL / 'adapter'
BASELINE_HEAD = BASELINE_MODEL / 'binary_classification_head.pt'
BASELINE_RESULTS = RESULT_ROOT / 'original'
FULL_MODELS, FULL_RESULTS = MODEL_ROOT / 'full_retraining', RESULT_ROOT / 'full_retraining'
UNLEARNING_RESULTS = RESULT_ROOT / 'unlearning'
METHOD_MODEL_ROOTS = {m: MODEL_ROOT / m for m in
    ['retain_set_fine_tuning', 'gradient_ascent', 'gradient_difference']}
assert BASELINE_MODEL not in METHOD_MODEL_ROOTS.values() and FULL_MODELS not in METHOD_MODEL_ROOTS.values()

SCENARIOS = ['recipient_withdrawal', 'donor_withdrawal', 'invalid_consent',
             'hospital_removal', 'retention_expiry']
SCENARIO_LABELS = {s: s.replace('_', ' ').title() for s in SCENARIOS}
EXPECTED_FORGET = {'recipient_withdrawal': 426, 'donor_withdrawal': 1992,
    'invalid_consent': 4148, 'hospital_removal': 4314, 'retention_expiry': 6262}
METHOD_CONFIGS = {
 'retain_set_fine_tuning': {'learning_rate': 1e-4, 'weight_decay': 1e-4, 'batch_size': 8,
    'gradient_accumulation': 4, 'maximum_epochs': 30, 'patience': 5,
    'selection': 'minimum retained-validation weighted BCE', 'mlp_source': '03_retain_set_fine_tuning.ipynb'},
 'gradient_ascent': {'learning_rate': 1e-5, 'weight_decay': 0.0, 'batch_size': 8,
    'gradient_accumulation': 4, 'maximum_epochs': 20, 'safety_patience': 3,
    'gradient_clip_norm': 1.0, 'validation_relative_allowance': 0.05,
    'selection': 'largest complete forget BCE inside retained-validation boundary',
    'mlp_source': '04_gradient_ascent.ipynb'},
 'gradient_difference': {'learning_rate': 1e-5, 'weight_decay': 0.0, 'batch_size': 8,
    'gradient_accumulation': 4, 'epochs': 5, 'gradient_clip_norm': 1.0,
    'forget_weight_lambda': 1.0, 'selection': 'fixed final epoch; no model selection',
    'mlp_source': '06_gradient_difference.ipynb'},
}
display(pd.DataFrame(METHOD_CONFIGS).T)

,learning_rate,weight_decay,batch_size,gradient_accumulation,maximum_epochs,patience,selection,mlp_source,safety_patience,gradient_clip_norm,validation_relative_allowance,epochs,forget_weight_lambda
retain_set_fine_tuning,0.0001,0.0001,8,4,30,5,minimum retained-validation weighted BCE,03_retain_set_fine_tuning.ipynb,NaN,NaN,NaN,NaN,NaN
gradient_ascent,0.00001,0.0,8,4,20,NaN,largest complete forget BCE inside retained-va...,04_gradient_ascent.ipynb,3,1.0,0.05,NaN,NaN
gradient_difference,0.00001,0.0,8,4,NaN,NaN,fixed final epoch; no model selection,06_gradient_difference.ipynb,NaN,1.0,NaN,5,1.0


# 3. Load Frozen Original Qwen

## 3.1 Baseline Identity

The authoritative baseline is run `20260829T151430Z`: `unsloth/Qwen3.5-2B-Base`, selected epoch 6, LoRA rank/alpha 16/16, and frozen threshold 0.55.

## 3.2 Baseline Artefacts

The pretrained base is reconstructed, the authoritative saved PEFT adapter is loaded as trainable, and the separately saved binary head is restored last. The frozen files are never write targets.

In [43]:
baseline_files = {
 'adapter_config': BASELINE_ADAPTER / 'adapter_config.json',
 'adapter_weights': BASELINE_ADAPTER / 'adapter_model.safetensors',
 'processor': BASELINE_ADAPTER / 'processor_config.json',
 'tokenizer': BASELINE_ADAPTER / 'tokenizer.json',
 'head': BASELINE_HEAD,
 'configuration': BASELINE_RESULTS / 'experiment_configuration.json',
 'threshold': BASELINE_RESULTS / 'selected_threshold.json',
 'serialisation': BASELINE_RESULTS / 'serialisation_specification.json',
 'metrics': BASELINE_RESULTS / 'retained_test_metrics.csv',
 'predictions': BASELINE_RESULTS / 'retained_test_probabilities.csv'}
missing = [path for path in baseline_files.values() if not path.is_file()]
if missing: raise FileNotFoundError('Incomplete frozen Original Qwen bundle: ' + '; '.join(map(str, missing)))
baseline_config = json.loads(baseline_files['configuration'].read_text())
adapter_config = json.loads(baseline_files['adapter_config'].read_text())
serialisation = json.loads(baseline_files['serialisation'].read_text())
baseline_predictions = pd.read_csv(baseline_files['predictions'])
assert baseline_config['run_id'] == '20260829T151430Z'
assert baseline_config['model_id'] == adapter_config['base_model_name_or_path']
assert baseline_config['best_epoch'] == 6 and baseline_config['selected_threshold'] == THRESHOLD
assert len(baseline_predictions) == 8988 and baseline_predictions.assessment_id.is_unique
MODEL_ID, MAX_LENGTH = baseline_config['model_id'], int(baseline_config['max_seq_length'])
PROTECTED_PATHS = list(baseline_files.values())
for scenario in SCENARIOS:
    PROTECTED_PATHS += list((FULL_MODELS / scenario / 'adapter').glob('*'))
    PROTECTED_PATHS += [FULL_MODELS / scenario / 'binary_classification_head.pt']
    PROTECTED_PATHS += list((FULL_RESULTS / scenario).glob('*'))
def frozen_state():
    return {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in PROTECTED_PATHS if p.is_file()}
FROZEN_STATE_BEFORE = frozen_state()
display(pd.DataFrame([{'Artefact': k, 'Exists': v.is_file()} for k, v in baseline_files.items()]))

,Artefact,Exists
0,adapter_config,True
1,adapter_weights,True
2,processor,True
3,tokenizer,True
4,head,True
5,configuration,True
6,threshold,True
7,serialisation,True
8,metrics,True
9,predictions,True


In [44]:
class FP32ClassificationHead(nn.Linear):
    def forward(self, hidden_states): return F.linear(hidden_states.float(), self.weight, self.bias)

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def load_fresh_original_qwen(show_summary=True):
    reset_seed()
    # Text-only classification does not need Qwen image/video processors.
    # Load the exact saved tokenizer directly, avoiding optional torchvision/video dependencies.
    tokenizer = AutoTokenizer.from_pretrained(BASELINE_ADAPTER, local_files_only=True)
    tokenizer.padding_side = 'right'
    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
    base = Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_ID, dtype=MODEL_DTYPE, low_cpu_mem_usage=True)
    old_head = base.get_output_embeddings()
    base.set_output_embeddings(FP32ClassificationHead(old_head.in_features, 2, bias=False, dtype=torch.float32))
    base.config.num_labels, base.config.pad_token_id = 2, tokenizer.pad_token_id
    # PEFT reads the saved LoRA configuration and weights; no new LoRA configuration is created.
    model = PeftModel.from_pretrained(base, BASELINE_ADAPTER, is_trainable=True)
    # Restore the separately saved binary head last so no random head survives.
    head_state = torch.load(BASELINE_HEAD, map_location='cpu', weights_only=True)
    current = model.state_dict()
    assert head_state and all(name in current and current[name].shape == value.shape for name, value in head_state.items())
    model.load_state_dict(head_state, strict=False)
    model.to(DEVICE); model.config.use_cache = False
    for parameter in model.parameters():
        if parameter.requires_grad: parameter.data = parameter.data.float()
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if show_summary:
        display(pd.Series({'Total parameters': total, 'Trainable parameters': trainable,
            'Trainable percentage': 100 * trainable / total, 'Device': str(DEVICE)}).to_frame('Value'))
    assert trainable > 0 and trainable < total
    return model, tokenizer

def clear_device_cache():
    gc.collect()
    if DEVICE.type == 'mps' and hasattr(torch, 'mps'): torch.mps.empty_cache()
    elif DEVICE.type == 'cuda': torch.cuda.empty_cache()

# 4. Full Retraining References

Full Retraining is a frozen post-hoc reference, not an optimisation target. The question is whether an approximate model behaves like the model that never trained on the deleted examples—not merely whether it performs badly on them. These historical references were produced in the original A100/CUDA/Unsloth environment. No local Full Retraining result is available, so the established frozen reference is retained and the execution-environment difference is treated as a limitation when interpreting absolute probability distributions.

In [45]:
FULL_FILES = ['COMPLETE.json', 'retained_test_metrics.csv', 'retained_test_probabilities.csv',
              'forget_set_probabilities.csv', 'training_history.csv']
full_retraining = {}
for scenario in SCENARIOS:
    folder = FULL_RESULTS / scenario
    files = {name: folder / name for name in FULL_FILES}
    if not all(path.is_file() for path in files.values()): raise FileNotFoundError(f'Incomplete Full Retraining reference: {folder}')
    complete = json.loads(files['COMPLETE.json'].read_text())
    assert complete['status'] == 'complete' and complete['training_forget_rows'] == EXPECTED_FORGET[scenario]
    full_retraining[scenario] = {'complete': complete,
        'metrics': pd.read_csv(files['retained_test_metrics.csv']).iloc[0],
        'retained': pd.read_csv(files['retained_test_probabilities.csv']),
        'forget': pd.read_csv(files['forget_set_probabilities.csv']),
        'history': pd.read_csv(files['training_history.csv'])}
    full_retraining[scenario]['retained']['assessment_id'] = full_retraining[scenario]['retained'].assessment_id.astype(str)
    full_retraining[scenario]['forget']['assessment_id'] = full_retraining[scenario]['forget'].assessment_id.astype(str)
display(pd.DataFrame([{'Scenario': SCENARIO_LABELS[s], 'Retained rows': len(full_retraining[s]['retained']),
    'Forget rows': len(full_retraining[s]['forget'])} for s in SCENARIOS]))

,Scenario,Retained rows,Forget rows
0,Recipient Withdrawal,8898,426
1,Donor Withdrawal,8496,1992
2,Invalid Consent,8078,4148
3,Hospital Removal,8124,4314
4,Retention Expiry,7614,6262


# 5. Deletion Scenarios

The authoritative scenario-membership file supplies the training forget set and deleted validation/test memberships. No scenario is regenerated or sampled anew.

In [46]:
DATA = FINAL / 'data' / 'final' / 'kidney_transplant_assessments.csv'
FEATURES = FINAL / 'data' / 'final' / 'classifier_feature_list.json'
SPLITS = FINAL / 'processed_data' / 'split_assignments.csv'
MEMBERSHIP = FINAL / 'processed_data' / 'deletion_scenario_membership.csv'
assessments, split_map, membership = pd.read_csv(DATA), pd.read_csv(SPLITS), pd.read_csv(MEMBERSHIP)
contract = json.loads(FEATURES.read_text()); target = contract['target']; features = contract['classifier_features']
data = assessments.merge(split_map[['recipient_id', 'donor_id', 'split']],
    on=['recipient_id', 'donor_id'], validate='many_to_one')
labels = serialisation['display_labels']; binary = {'previous_transplant', 'infection_indicator', 'previous_rejection'}
def formatted(feature, value):
    if pd.isna(value): return 'missing'
    if feature in binary: return 'yes' if int(value) == 1 else 'no'
    if isinstance(value, (float, np.floating)): return f'{float(value):.4f}'.rstrip('0').rstrip('.')
    return str(value).strip()
data['text'] = data.apply(lambda row: '\n'.join(f'{labels[f]}: {formatted(f, row[f])}.' for f in features), axis=1)
data['label'] = data[target].astype('int64'); data['assessment_id'] = data['assessment_id'].astype(str)
parts_by_split = {name: data.loc[data.split.eq(name)].copy() for name in ['train', 'validation', 'test']}
scenario_sets = {}
for scenario in SCENARIOS:
    current = membership.loc[membership.scenario.eq(scenario)].copy(); current['assessment_id'] = current.assessment_id.astype(str)
    ids = {kind: set(current.loc[current.membership_type.eq(kind), 'assessment_id'])
           for kind in ['training_forget', 'deleted_validation', 'deleted_test']}
    scenario_sets[scenario] = {'training_forget': parts_by_split['train'].loc[parts_by_split['train'].assessment_id.isin(ids['training_forget'])].copy(),
      'retained_train': parts_by_split['train'].loc[~parts_by_split['train'].assessment_id.isin(ids['training_forget'])].copy(),
      'retained_validation': parts_by_split['validation'].loc[~parts_by_split['validation'].assessment_id.isin(ids['deleted_validation'])].copy(),
      'retained_test': parts_by_split['test'].loc[~parts_by_split['test'].assessment_id.isin(ids['deleted_test'])].copy()}
    assert len(scenario_sets[scenario]['training_forget']) == EXPECTED_FORGET[scenario]
    assert len(scenario_sets[scenario]['retained_test']) == full_retraining[scenario]['complete']['retained_test_rows']
display(pd.DataFrame([{'Scenario': SCENARIO_LABELS[s], **{k: len(v) for k, v in scenario_sets[s].items()}} for s in SCENARIOS]))

,Scenario,training_forget,retained_train,retained_validation,retained_test
0,Recipient Withdrawal,426,41598,8904,8898
1,Donor Withdrawal,1992,40032,8472,8496
2,Invalid Consent,4148,37876,8046,8078
3,Hospital Removal,4314,37710,8166,8124
4,Retention Expiry,6262,35762,7624,7614


# 6. Shared Evaluation

## 6.1 Utility
All retained utility uses the frozen threshold 0.55.

## 6.2 Truth Ratio
Truth Ratio is `(p_incorrect + epsilon) / (p_true + epsilon)`.

## 6.3 KS Test
Approximate and matching Full Retraining Truth Ratio distributions are aligned by assessment ID and compared with a two-sample KS test.

## 6.4 Runtime Comparison
Speed-up is saved Full Retraining training time divided by approximate update time. MPS peak memory is not fabricated.

In [47]:
UTILITY_KEYS = ['pr_auc', 'balanced_accuracy', 'binary_cross_entropy', 'f1', 'auroc', 'precision', 'recall', 'specificity']
def utility_metrics(frame):
    y = frame.label.to_numpy(); p = frame.probability_class_1.to_numpy(); pred = (p >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'n': len(y), 'pr_auc': average_precision_score(y, p),
      'balanced_accuracy': balanced_accuracy_score(y, pred), 'binary_cross_entropy': log_loss(y, p, labels=[0, 1]),
      'f1': f1_score(y, pred, zero_division=0), 'auroc': roc_auc_score(y, p),
      'precision': precision_score(y, pred, zero_division=0), 'recall': recall_score(y, pred, zero_division=0),
      'specificity': tn / (tn + fp), 'threshold': THRESHOLD}
def truth_components(labels, probabilities, epsilon=1e-12):
    p_true = np.where(labels == 1, probabilities, 1 - probabilities); p_incorrect = 1 - p_true
    return p_true, p_incorrect, (p_incorrect + epsilon) / (p_true + epsilon)
def forgetting_evidence(approximate, reference):
    joined = approximate.merge(reference, on='assessment_id', suffixes=('_approximate', '_full'), validate='one_to_one')
    assert len(joined) == len(approximate) == len(reference)
    assert np.array_equal(joined.label_approximate, joined.label_full)
    y = joined.label_approximate.to_numpy(); ap = joined.probability_class_1_approximate.to_numpy(); fp = joined.probability_class_1_full.to_numpy()
    at, ai, ar = truth_components(y, ap); ft, fi, fr = truth_components(y, fp)
    ks = ks_2samp(ar, fr, alternative='two-sided', method='auto')
    values = pd.DataFrame({'assessment_id': joined.assessment_id, 'label': y,
      'approximate_probability': ap, 'full_retraining_probability': fp,
      'approximate_p_true': at, 'approximate_p_incorrect': ai, 'approximate_truth_ratio': ar,
      'full_p_true': ft, 'full_p_incorrect': fi, 'full_truth_ratio': fr, 'epsilon': 1e-12})
    return values, {'forget_rows': len(values), 'ks_statistic': float(ks.statistic), 'ks_p_value': float(ks.pvalue)}

baseline_predictions['assessment_id'] = baseline_predictions.assessment_id.astype(str)
scenario_original = {}
for scenario in SCENARIOS:
    ids = scenario_sets[scenario]['retained_test'][['assessment_id', 'label']]
    matched = ids.merge(baseline_predictions[['assessment_id', 'label', 'probability_class_1']],
        on='assessment_id', suffixes=('_scenario', ''), validate='one_to_one')
    assert len(matched) == len(ids) and np.array_equal(matched.label_scenario, matched.label)
    scenario_original[scenario] = utility_metrics(matched[['assessment_id', 'label', 'probability_class_1']])

In [48]:
class TextDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]; return {'text': row.text, 'label': int(row.label), 'assessment_id': row.assessment_id}
def collate_text(rows, tokenizer):
    encoded = tokenizer([r['text'] for r in rows], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    return {'inputs': encoded, 'labels': torch.tensor([r['label'] for r in rows]), 'ids': [r['assessment_id'] for r in rows]}
def make_loader(frame, tokenizer, batch_size, shuffle=False, seed=SEED):
    return DataLoader(TextDataset(frame), batch_size=batch_size, shuffle=shuffle, num_workers=0,
        generator=torch.Generator().manual_seed(seed), collate_fn=lambda rows: collate_text(rows, tokenizer))
def final_logits(model, batch):
    inputs = {k: v.to(DEVICE) for k, v in batch['inputs'].items()}
    sequence = model(**inputs).logits; indices = inputs['attention_mask'].sum(1) - 1
    return sequence[torch.arange(len(indices), device=DEVICE), indices]
def predict_frame(model, tokenizer, frame, description):
    model.eval(); rows = []
    with torch.no_grad():
        for batch in tqdm(make_loader(frame, tokenizer, 8), desc=description):
            logits = final_logits(model, batch).float(); probability = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            rows.extend({'assessment_id': i, 'label': int(y), 'probability_class_1': float(p),
                'prediction': int(p >= THRESHOLD)} for i, y, p in zip(batch['ids'], batch['labels'], probability))
    return pd.DataFrame(rows)
def positive_weight(frame):
    positives = int(frame.label.sum()); return float((len(frame) - positives) / positives)
def batch_ce(logits, labels, pos_weight=None):
    if pos_weight is None: return F.cross_entropy(logits.float(), labels)
    weights = torch.tensor([1.0, pos_weight], device=logits.device)
    return F.cross_entropy(logits.float(), labels, weight=weights, reduction='sum') / len(labels)
def frame_loss(model, tokenizer, frame, pos_weight=None):
    model.eval(); total = 0.0
    with torch.no_grad():
        for batch in make_loader(frame, tokenizer, 8):
            labels = batch['labels'].to(DEVICE); total += float(batch_ce(final_logits(model, batch), labels, pos_weight)) * len(labels)
    return total / len(frame)
def capture_trainable(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
def restore_trainable(model, state):
    named = dict(model.named_parameters())
    with torch.no_grad():
        for name, value in state.items(): named[name].copy_(value.to(named[name].device))

## 3.3 Historical Baseline Reproduction Check

The Original Qwen model was created on RunPod using an NVIDIA A100, CUDA, and Unsloth. This notebook reconstructs the same saved base-model identity, PEFT/LoRA adapter, binary classification head, tokenizer, serialisation, and threshold under Mac/MPS Transformers and PEFT. Similar but non-identical probabilities are expected across these execution paths. The strict comparison with the frozen A100 predictions is retained as cross-environment reproducibility documentation only; it is not an execution gate.

## 3.4 Local Experimental Baseline

The reconstructed classifier produced in the current environment is frozen as a separate local baseline for this approximate-unlearning extension. It does not replace or modify the historical A100 baseline. The local baseline uses the same saved artefacts and current execution environment, and every method–scenario run must independently reproduce this exact local start state.

## Why a Local Baseline Is Used

The original Qwen classifier was trained and evaluated using an NVIDIA A100, CUDA and Unsloth. Reconstructing the same saved model artefacts under the later Mac/MPS Transformers and PEFT environment produced small but non-negligible differences in predicted probabilities. The local reconstruction was therefore not treated as numerically identical to the historical model.

To keep the approximate-unlearning comparison controlled, the reconstructed model is instead frozen as a new local experimental baseline. Each Retain-Set Fine-Tuning, Gradient Ascent and Gradient Difference run begins independently from this same verified local state. The historical A100 comparison is retained as a reproducibility check, but it does not control whether the local approximate-unlearning experiment can run.

In [49]:
VERIFICATION_ATOL = 1e-5
VERIFICATION_RTOL = 1e-4
LOCAL_BASELINE_DIR = UNLEARNING_RESULTS / 'local_experimental_baseline'
LOCAL_BASELINE_DIR.mkdir(parents=True, exist_ok=True)

def comparison_report(observed, expected):
    aligned_ids = observed.assessment_id.tolist() == expected.assessment_id.tolist()
    labels_align = np.array_equal(observed.label, expected.label)
    differences = np.abs(observed.probability_class_1.to_numpy() - expected.probability_class_1.to_numpy())
    passed = aligned_ids and labels_align and np.allclose(
        observed.probability_class_1, expected.probability_class_1,
        atol=VERIFICATION_ATOL, rtol=VERIFICATION_RTOL)
    return {'compared_rows': len(observed), 'ids_aligned': aligned_ids, 'labels_aligned': labels_align,
        'maximum_absolute_difference': float(differences.max()),
        'mean_absolute_difference': float(differences.mean()),
        'absolute_tolerance': VERIFICATION_ATOL, 'relative_tolerance': VERIFICATION_RTOL,
        'status': 'PASS' if passed else 'FAIL', 'passed': bool(passed)}

def state_fingerprint(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode('utf-8')); digest.update(str(tensor.dtype).encode('ascii'))
        digest.update(str(tuple(tensor.shape)).encode('ascii')); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

# Construct once, document historical reproduction, then freeze the current local state.
LOCAL_BASELINE_MODEL, LOCAL_BASELINE_TOKENIZER = load_fresh_original_qwen()
LOCAL_BASELINE_MODEL.eval()
historical_expected = baseline_predictions.head(64).copy()
LOCAL_BASELINE_PROBE_ROWS = (parts_by_split['test'].set_index('assessment_id')
    .loc[historical_expected.assessment_id].reset_index())
LOCAL_BASELINE_PROBE_PREDICTIONS = predict_frame(
    LOCAL_BASELINE_MODEL, LOCAL_BASELINE_TOKENIZER, LOCAL_BASELINE_PROBE_ROWS, 'Historical A100 reproduction probe')
HISTORICAL_BASELINE_VERIFICATION = comparison_report(LOCAL_BASELINE_PROBE_PREDICTIONS, historical_expected)
HISTORICAL_BASELINE_REPRODUCED = bool(HISTORICAL_BASELINE_VERIFICATION['passed'])
display(pd.Series(HISTORICAL_BASELINE_VERIFICATION).to_frame('Historical A100 comparison'))
if not HISTORICAL_BASELINE_REPRODUCED:
    display(Markdown(
        '> **Historical reproduction status: FAIL / not numerically identical.** IDs and labels remain aligned, '
        'but the A100/CUDA/Unsloth probabilities are not reproduced within the unchanged strict tolerance. '
        'This is documented for transparency and does not suppress the local experiment.'))

LOCAL_BASELINE_STATE = capture_trainable(LOCAL_BASELINE_MODEL)
LOCAL_BASELINE_HEAD_STATE = {name: value.detach().cpu().clone()
    for name, value in LOCAL_BASELINE_MODEL.state_dict().items() if 'lm_head' in name}
if not LOCAL_BASELINE_STATE or not LOCAL_BASELINE_HEAD_STATE:
    raise RuntimeError('The local baseline trainable state or binary classification head could not be frozen.')
LOCAL_BASELINE_CONFIGURATION = {
    'purpose': 'local approximate-unlearning start state only',
    'historical_run_id': baseline_config['run_id'], 'base_model_id': MODEL_ID,
    'adapter_path': str(BASELINE_ADAPTER), 'head_path': str(BASELINE_HEAD),
    'tokenizer_path': str(BASELINE_ADAPTER),
    'serialisation_specification_path': str(baseline_files['serialisation']),
    'serialisation_specification_sha256': hashlib.sha256(baseline_files['serialisation'].read_bytes()).hexdigest(),
    'threshold': THRESHOLD, 'maximum_length': MAX_LENGTH, 'device_type': DEVICE.type,
    'model_dtype': str(MODEL_DTYPE), 'seed': SEED,
    'trainable_state_sha256': state_fingerprint(LOCAL_BASELINE_STATE),
    'head_state_sha256': state_fingerprint(LOCAL_BASELINE_HEAD_STATE)}
LOCAL_BASELINE_RETAINED_TEST_PREDICTIONS = predict_frame(
    LOCAL_BASELINE_MODEL, LOCAL_BASELINE_TOKENIZER, parts_by_split['test'], 'Freeze local retained-test baseline')
LOCAL_BASELINE_RETAINED_TEST_METRICS = utility_metrics(LOCAL_BASELINE_RETAINED_TEST_PREDICTIONS)
LOCAL_SCENARIO_BASELINE_METRICS = {}
for scenario in SCENARIOS:
    retained_ids = scenario_sets[scenario]['retained_test'][['assessment_id', 'label']]
    local_matched = retained_ids.merge(
        LOCAL_BASELINE_RETAINED_TEST_PREDICTIONS[['assessment_id', 'label', 'probability_class_1']],
        on='assessment_id', suffixes=('_scenario', ''), validate='one_to_one')
    if len(local_matched) != len(retained_ids) or not np.array_equal(local_matched.label_scenario, local_matched.label):
        raise RuntimeError(f'Local retained-test alignment failed for {scenario}.')
    LOCAL_SCENARIO_BASELINE_METRICS[scenario] = utility_metrics(
        local_matched[['assessment_id', 'label', 'probability_class_1']])
LOCAL_BASELINE_PROBE_PREDICTIONS.to_csv(LOCAL_BASELINE_DIR / 'probe_predictions.csv', index=False)
LOCAL_BASELINE_RETAINED_TEST_PREDICTIONS.to_csv(LOCAL_BASELINE_DIR / 'retained_test_probabilities.csv', index=False)
pd.DataFrame([LOCAL_BASELINE_RETAINED_TEST_METRICS]).to_csv(LOCAL_BASELINE_DIR / 'retained_test_metrics.csv', index=False)
(LOCAL_BASELINE_DIR / 'manifest.json').write_text(json.dumps(LOCAL_BASELINE_CONFIGURATION, indent=2), encoding='utf-8')
LOCAL_BASELINE_FROZEN = True
del LOCAL_BASELINE_MODEL, LOCAL_BASELINE_TOKENIZER; clear_device_cache()

def restore_local_baseline(model):
    restore_trainable(model, LOCAL_BASELINE_STATE)
    model.load_state_dict(LOCAL_BASELINE_HEAD_STATE, strict=False)
    model.eval()
    return model

RUN_START_AUDIT = []
def fresh_local_baseline(method=None, scenario=None, verify_start=True):
    model, tokenizer = load_fresh_original_qwen(show_summary=False)
    restore_local_baseline(model)
    if verify_start:
        expected = LOCAL_BASELINE_PROBE_PREDICTIONS.head(8).copy()
        probe = LOCAL_BASELINE_PROBE_ROWS.head(8).copy()
        observed = predict_frame(model, tokenizer, probe, f'Verify local start: {method} / {scenario}')
        report = comparison_report(observed, expected)
        RUN_START_AUDIT.append({'method': method, 'scenario': scenario,
            'state_sha256': state_fingerprint(capture_trainable(model)), **report})
        if not report['passed'] or RUN_START_AUDIT[-1]['state_sha256'] != LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256']:
            del model, tokenizer; clear_device_cache()
            raise RuntimeError(f'Local start-state verification failed for {method} / {scenario}.')
    return model, tokenizer

def verify_local_baseline():
    model, tokenizer = fresh_local_baseline(verify_start=False)
    try:
        observed = predict_frame(model, tokenizer, LOCAL_BASELINE_PROBE_ROWS, 'Verify frozen local baseline')
        report = comparison_report(observed, LOCAL_BASELINE_PROBE_PREDICTIONS)
        report['state_sha256_matches'] = (
            state_fingerprint(capture_trainable(model)) == LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256'])
        report['passed'] = bool(report['passed'] and report['state_sha256_matches'])
        report['status'] = 'PASS' if report['passed'] else 'FAIL'
        display(pd.Series(report).to_frame('Local baseline reproduction'))
        return report
    finally:
        del model, tokenizer; clear_device_cache()

LOCAL_BASELINE_VERIFICATION = verify_local_baseline() if RUN_FINAL_EXPERIMENT else None
LOCAL_BASELINE_VERIFICATION_PASSED = bool(LOCAL_BASELINE_VERIFICATION and LOCAL_BASELINE_VERIFICATION['passed'])
if RUN_FINAL_EXPERIMENT and not LOCAL_BASELINE_VERIFICATION_PASSED:
    raise RuntimeError('The frozen local experimental baseline is not reproducible in the current environment.')
APPROXIMATE_UNLEARNING_EXECUTED = False
APPROXIMATE_EXECUTION_ATTEMPTS = []
APPROXIMATE_RUN_REQUESTS = []
LLM_APPROX_UNLEARNING_STATUS = (
    'ready_local_baseline_verified' if LOCAL_BASELINE_VERIFICATION_PASSED
    else 'not_run_local_verification_disabled')

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/opt/anaconda3/envs/msc-unlearning/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


,Value
Total parameters,2236360512
Trainable parameters,23114752
Trainable percentage,1.033588
Device,mps


Historical A100 reproduction probe:   0%|          | 0/8 [00:00<?, ?it/s]

,Historical A100 comparison
compared_rows,64
ids_aligned,True
labels_aligned,True
maximum_absolute_difference,0.038142
mean_absolute_difference,0.010629
absolute_tolerance,0.00001
relative_tolerance,0.0001
status,FAIL
passed,False


...

Freeze local retained-test baseline:   0%|          | 0/1124 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/opt/anaconda3/envs/msc-unlearning/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Verify frozen local baseline:   0%|          | 0/8 [00:00<?, ?it/s]

,Local baseline reproduction
compared_rows,64
ids_aligned,True
labels_aligned,True
maximum_absolute_difference,0.0
mean_absolute_difference,0.0
absolute_tolerance,0.00001
relative_tolerance,0.0001
status,PASS
passed,True
state_sha256_matches,True


## 3.5 Baseline Interpretation

The historical A100 result and the local experimental baseline are deliberately separate. Failure of the historical reproduction check documents a cross-environment limitation only. Equality is mandatory before each local unlearning run; prediction changes after an unlearning update are expected experimental outcomes and are never treated as start-state failures.

In [50]:
FULL_RETRAINING_AVAILABLE = (set(full_retraining) == set(SCENARIOS) and
    all(full_retraining[s]['complete'].get('status') == 'complete' for s in SCENARIOS))
component_status = pd.DataFrame([
    {'Component': 'Historical A100 Baseline', 'Status': 'Available',
     'Reason': 'Existing immutable authoritative baseline'},
    {'Component': 'Historical reproduction on Mac',
     'Status': 'Reproduced' if HISTORICAL_BASELINE_REPRODUCED else 'Failed / not numerically identical',
     'Reason': 'Informational cross-environment check; never an execution gate'},
    {'Component': 'Local Experimental Baseline',
     'Status': 'Verified' if LOCAL_BASELINE_VERIFICATION_PASSED else 'Failed',
     'Reason': 'Independent reconstruction matches the newly frozen local state'},
    {'Component': 'Full Retraining', 'Status': 'Available' if FULL_RETRAINING_AVAILABLE else 'Unavailable',
     'Reason': 'Historical A100 reference; environment limitation documented'},
    *[{'Component': label, 'Status': 'Eligible' if LOCAL_BASELINE_VERIFICATION_PASSED else 'Blocked',
       'Reason': 'Will start independently from verified local baseline'}
      for label in ['Retain-Set Fine-Tuning', 'Gradient Ascent', 'Gradient Difference']]
])
display(component_status)
STATUS_PATH = RESULT_ROOT / 'qwen_approximate_unlearning_status.json'
def current_status_payload():
    historical = HISTORICAL_BASELINE_VERIFICATION or {}
    local = LOCAL_BASELINE_VERIFICATION or {}
    return {
        'status': LLM_APPROX_UNLEARNING_STATUS,
        'historical_baseline_available': True,
        'historical_baseline_reproduced': HISTORICAL_BASELINE_REPRODUCED,
        'historical_ids_aligned': historical.get('ids_aligned'),
        'historical_labels_aligned': historical.get('labels_aligned'),
        'historical_max_probability_difference': historical.get('maximum_absolute_difference'),
        'historical_mean_probability_difference': historical.get('mean_absolute_difference'),
        'historical_check_is_execution_gate': False,
        'local_baseline_frozen': LOCAL_BASELINE_FROZEN,
        'local_baseline_verification_passed': LOCAL_BASELINE_VERIFICATION_PASSED,
        'local_baseline_state_sha256': LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256'],
        'local_max_probability_difference': local.get('maximum_absolute_difference'),
        'local_mean_probability_difference': local.get('mean_absolute_difference'),
        'tolerance': {'absolute': VERIFICATION_ATOL, 'relative': VERIFICATION_RTOL},
        'approximate_unlearning_executed': APPROXIMATE_UNLEARNING_EXECUTED,
        'full_retraining_available': FULL_RETRAINING_AVAILABLE,
        'full_retraining_environment': 'historical A100/CUDA/Unsloth reference',
    }
STATUS_PATH.write_text(json.dumps(current_status_payload(), indent=2), encoding='utf-8')
print('Machine-readable status:', STATUS_PATH)

,Component,Status,Reason
0,Historical A100 Baseline,Available,Existing immutable authoritative baseline
1,Historical reproduction on Mac,Failed / not numerically identical,Informational cross-environment check; never a...
2,Local Experimental Baseline,Verified,Independent reconstruction matches the newly f...
3,Full Retraining,Available,Historical A100 reference; environment limitat...
4,Retain-Set Fine-Tuning,Eligible,Will start independently from verified local b...
5,Gradient Ascent,Eligible,Will start independently from verified local b...
6,Gradient Difference,Eligible,Will start independently from verified local b...


Machine-readable status: /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission/results/qwen/qwen_approximate_unlearning_status.json


In [51]:
def run_paths(method, scenario):
    model = METHOD_MODEL_ROOTS[method] / scenario; result = UNLEARNING_RESULTS / method / scenario
    return {'model': model, 'adapter': model / 'adapter', 'head': model / 'binary_classification_head.pt',
      'result': result, 'complete': result / 'COMPLETE.json'}
def required_run_files(method, scenario):
    p = run_paths(method, scenario)
    return [p['adapter'] / 'adapter_config.json', p['adapter'] / 'adapter_model.safetensors', p['head'],
      p['result'] / 'training_history.csv', p['result'] / 'configuration.json', p['result'] / 'runtime.json',
      p['result'] / 'retained_test_metrics.csv', p['result'] / 'retained_test_probabilities.csv',
      p['result'] / 'forget_set_probabilities.csv', p['result'] / 'truth_ratio_values.csv',
      p['result'] / 'forgetting_metrics.csv']
def prepare_run(method, scenario):
    p = run_paths(method, scenario)
    if p['complete'].is_file():
        complete = json.loads(p['complete'].read_text())
        missing = [path for path in required_run_files(method, scenario) if not path.is_file()]
        if complete.get('status') == 'complete' and not missing: return 'skip'
        raise RuntimeError(f'Invalid completed run: {p["result"]}; missing={missing}')
    if p['model'].exists() or p['result'].exists(): raise RuntimeError(f'Partial run found; inspect without overwriting: {p}')
    p['model'].mkdir(parents=True); p['result'].mkdir(parents=True); return 'run'
def save_completed(method, scenario, model, tokenizer, history, summary, retained, forget, truth, forgetting):
    p = run_paths(method, scenario); config = METHOD_CONFIGS[method]
    model.save_pretrained(p['adapter'], safe_serialization=True); tokenizer.save_pretrained(p['adapter'])
    head = {name: value.detach().cpu() for name, value in model.state_dict().items() if 'lm_head' in name}
    if not head: raise RuntimeError('Binary head not found while saving.')
    torch.save(head, p['head'])
    pd.DataFrame(history).to_csv(p['result'] / 'training_history.csv', index=False)
    (p['result'] / 'configuration.json').write_text(json.dumps({**config, **summary}, indent=2))
    pd.DataFrame([utility_metrics(retained)]).to_csv(p['result'] / 'retained_test_metrics.csv', index=False)
    retained.to_csv(p['result'] / 'retained_test_probabilities.csv', index=False)
    forget.to_csv(p['result'] / 'forget_set_probabilities.csv', index=False)
    truth.to_csv(p['result'] / 'truth_ratio_values.csv', index=False)
    pd.DataFrame([forgetting]).to_csv(p['result'] / 'forgetting_metrics.csv', index=False)
    full_seconds = float(full_retraining[scenario]['complete']['training_seconds'])
    runtime = {'training_seconds': summary['training_seconds'], 'full_retraining_seconds': full_seconds,
      'speed_up': full_seconds / summary['training_seconds'], 'peak_memory_gib': None,
      'memory_note': 'MPS/CPU memory is not directly comparable with saved CUDA peak memory.'}
    (p['result'] / 'runtime.json').write_text(json.dumps(runtime, indent=2))
    missing = [path for path in required_run_files(method, scenario) if not path.is_file()]
    if missing: raise RuntimeError('Cannot complete; missing artefacts: ' + '; '.join(map(str, missing)))
    complete = {'status': 'complete', 'method': method, 'scenario': scenario, **summary, **forgetting, **runtime}
    p['complete'].write_text(json.dumps(complete, indent=2))
def evaluate_and_save(method, scenario, model, tokenizer, history, summary):
    retained = predict_frame(model, tokenizer, scenario_sets[scenario]['retained_test'], f'{method} retained')
    forget = predict_frame(model, tokenizer, scenario_sets[scenario]['training_forget'], f'{method} forget')
    truth, forgetting = forgetting_evidence(forget, full_retraining[scenario]['forget'])
    save_completed(method, scenario, model, tokenizer, history, summary, retained, forget, truth, forgetting)
def method_results(method):
    rows = []
    for scenario in SCENARIOS:
        p = run_paths(method, scenario)
        if not p['complete'].is_file(): continue
        complete = json.loads(p['complete'].read_text()); metrics = pd.read_csv(p['result'] / 'retained_test_metrics.csv').iloc[0]
        rows.append({'Scenario': SCENARIO_LABELS[scenario], **{k: metrics[k] for k in UTILITY_KEYS},
          'KS statistic': complete['ks_statistic'], 'KS p-value': complete['ks_p_value'],
          'Training seconds': complete['training_seconds'], 'Speed-up': complete['speed_up']})
    return pd.DataFrame(rows)

# 7. Retain-Set Fine-Tuning

## 7.1 Method
A fresh, independently verified copy of the local experimental Qwen baseline continues training only on retained training rows. The forget set is not an optimisation input.

## 7.2 Configuration
The final MLP rule is retained: AdamW, learning rate `1e-4`, weight decay `1e-4`, maximum 30 epochs, patience 5, and selection by minimum retained-validation weighted BCE. Retained-test and forget-set results never select a checkpoint. Batch 8 with accumulation 4 is the documented Qwen/MPS memory adaptation.

## 7.3 Training

In [52]:
def train_rsft(model, tokenizer, parts, scenario):
    cfg = METHOD_CONFIGS['retain_set_fine_tuning']; weight = positive_weight(parts['retained_train'])
    loader = make_loader(parts['retained_train'], tokenizer, cfg['batch_size'], True)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])
    best_loss, best_state, best_epoch, stale, history = np.inf, None, 0, 0, []
    started = time.perf_counter()
    for epoch in range(1, cfg['maximum_epochs'] + 1):
        model.train(); optimizer.zero_grad(set_to_none=True); total = 0.0
        for step, batch in enumerate(tqdm(loader, desc=f'RSFT {scenario} epoch {epoch}'), 1):
            y = batch['labels'].to(DEVICE); loss = batch_ce(final_logits(model, batch), y, weight)
            (loss / cfg['gradient_accumulation']).backward(); total += float(loss.detach()) * len(y)
            if step % cfg['gradient_accumulation'] == 0 or step == len(loader): optimizer.step(); optimizer.zero_grad(set_to_none=True)
        validation = frame_loss(model, tokenizer, parts['retained_validation'], weight)
        improved = validation < best_loss - 1e-12
        if improved: best_loss, best_state, best_epoch, stale = validation, capture_trainable(model), epoch, 0
        else: stale += 1
        history.append({'scenario': scenario, 'epoch': epoch, 'training_weighted_bce': total / len(parts['retained_train']),
            'validation_weighted_bce': validation, 'best_so_far': improved})
        if stale >= cfg['patience']: break
    restore_trainable(model, best_state)
    return model, history, {'selected_epoch': best_epoch, 'epochs_executed': epoch,
      'best_validation_weighted_bce': best_loss, 'training_seconds': time.perf_counter() - started}

In [53]:
if RUN_FINAL_EXPERIMENT and LOCAL_BASELINE_VERIFICATION_PASSED:
    for scenario in SCENARIOS:
        APPROXIMATE_RUN_REQUESTS.append(('retain_set_fine_tuning', scenario))
        action = prepare_run('retain_set_fine_tuning', scenario)
        if action == 'skip': print('RSFT skip:', scenario); continue
        APPROXIMATE_EXECUTION_ATTEMPTS.append(('retain_set_fine_tuning', scenario))
        APPROXIMATE_UNLEARNING_EXECUTED = True
        model, tokenizer = fresh_local_baseline('retain_set_fine_tuning', scenario)
        try:
            model, history, summary = train_rsft(model, tokenizer, scenario_sets[scenario], scenario)
            summary['started_from_verified_local_baseline'] = True
            summary['local_baseline_state_sha256'] = LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256']
            evaluate_and_save('retain_set_fine_tuning', scenario, model, tokenizer, history, summary)
        finally: del model, tokenizer; clear_device_cache()
else:
    if RUN_FINAL_EXPERIMENT: raise RuntimeError('Retain-Set Fine-Tuning requires a reproducible local baseline.')
    display(Markdown('**Retain-Set Fine-Tuning:** Execution disabled by RUN_FINAL_EXPERIMENT.'))

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/opt/anaconda3/envs/msc-unlearning/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Verify local start: retain_set_fine_tuning / recipient_withdrawal:   0%|          | 0/1 [00:00<?, ?it/s]

RSFT recipient_withdrawal epoch 1:   0%|          | 0/5200 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7.4 Utility Results
## 7.5 Forgetting Results
## 7.6 Runtime
## 7.7 Findings

The table keeps retained utility, KS evidence, and runtime separate. A smaller KS statistic means behaviour closer to Full Retraining; a high p-value alone is not proof of forgetting.

In [54]:
if LOCAL_BASELINE_VERIFICATION_PASSED:
    rsft_results = method_results('retain_set_fine_tuning')
    display(rsft_results.round(4))
else:
    rsft_results = pd.DataFrame(columns=['Scenario', *UTILITY_KEYS, 'KS statistic', 'KS p-value', 'Training seconds', 'Speed-up'])
    display(Markdown('No Retain-Set Fine-Tuning result table is shown because local execution is disabled.'))

""


# 8. Gradient Ascent

## 8.1 Method
A fresh, independently verified copy of the local experimental Qwen baseline maximises unweighted loss on training-forget rows only.

## 8.2 Configuration
The final MLP rule is retained: AdamW `1e-5`, zero weight decay, maximum 20 epochs, gradient clipping 1, a retained-validation weighted-loss allowance of 5%, and a three-breach safety stop. The eligible checkpoint with greatest complete forget BCE is selected; epoch 0 is restored if no update is eligible. Test, Full Retraining, and KS results never select the model. Batch 8 with accumulation 4 is the Qwen/MPS adaptation.

## 8.3 Training

In [55]:
def train_ga(model, tokenizer, parts, scenario):
    cfg = METHOD_CONFIGS['gradient_ascent']; weight = positive_weight(parts['retained_train'])
    loader = make_loader(parts['training_forget'], tokenizer, cfg['batch_size'], True)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=cfg['learning_rate'], weight_decay=0)
    baseline_state = capture_trainable(model); baseline_forget = frame_loss(model, tokenizer, parts['training_forget'])
    baseline_validation = frame_loss(model, tokenizer, parts['retained_validation'], weight)
    limit = baseline_validation * (1 + cfg['validation_relative_allowance'])
    best_state, best_forget, best_epoch, breaches, history = None, -np.inf, 0, 0, []
    started = time.perf_counter()
    for epoch in range(1, cfg['maximum_epochs'] + 1):
        model.train(); optimizer.zero_grad(set_to_none=True); total = 0.0
        for step, batch in enumerate(tqdm(loader, desc=f'GA {scenario} epoch {epoch}'), 1):
            y = batch['labels'].to(DEVICE); forget_loss = batch_ce(final_logits(model, batch), y)
            (-forget_loss / cfg['gradient_accumulation']).backward(); total += float(forget_loss.detach()) * len(y)
            if step % cfg['gradient_accumulation'] == 0 or step == len(loader):
                torch.nn.utils.clip_grad_norm_(parameters, cfg['gradient_clip_norm']); optimizer.step(); optimizer.zero_grad(set_to_none=True)
        complete_forget = frame_loss(model, tokenizer, parts['training_forget'])
        validation = frame_loss(model, tokenizer, parts['retained_validation'], weight)
        finite_parameters = all(bool(torch.isfinite(p).all()) for p in parameters)
        finite = bool(finite_parameters and np.isfinite([complete_forget, validation]).all())
        eligible = bool(finite and validation <= limit)
        history.append({'scenario': scenario, 'epoch': epoch, 'mean_batch_forget_bce': total / len(parts['training_forget']),
          'complete_forget_bce': complete_forget, 'retained_validation_weighted_bce': validation,
          'baseline_validation_weighted_bce': baseline_validation, 'validation_limit': limit, 'eligible': eligible})
        if eligible and complete_forget > best_forget: best_state, best_forget, best_epoch = capture_trainable(model), complete_forget, epoch
        breaches = 0 if eligible else breaches + 1
        if not finite or breaches >= cfg['safety_patience']: break
    restore_trainable(model, best_state if best_state is not None else baseline_state)
    return model, history, {'selected_epoch': best_epoch, 'eligible_updated_model': best_state is not None,
      'baseline_forget_bce': baseline_forget, 'selected_forget_bce': best_forget if best_state else baseline_forget,
      'baseline_validation_weighted_bce': baseline_validation, 'validation_limit': limit,
      'epochs_executed': epoch, 'training_seconds': time.perf_counter() - started}

In [56]:
if RUN_FINAL_EXPERIMENT and LOCAL_BASELINE_VERIFICATION_PASSED:
    for scenario in SCENARIOS:
        APPROXIMATE_RUN_REQUESTS.append(('gradient_ascent', scenario))
        action = prepare_run('gradient_ascent', scenario)
        if action == 'skip': print('GA skip:', scenario); continue
        APPROXIMATE_EXECUTION_ATTEMPTS.append(('gradient_ascent', scenario))
        APPROXIMATE_UNLEARNING_EXECUTED = True
        model, tokenizer = fresh_local_baseline('gradient_ascent', scenario)
        try:
            model, history, summary = train_ga(model, tokenizer, scenario_sets[scenario], scenario)
            summary['started_from_verified_local_baseline'] = True
            summary['local_baseline_state_sha256'] = LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256']
            evaluate_and_save('gradient_ascent', scenario, model, tokenizer, history, summary)
        finally: del model, tokenizer; clear_device_cache()
else:
    if RUN_FINAL_EXPERIMENT: raise RuntimeError('Gradient Ascent requires a reproducible local baseline.')
    display(Markdown('**Gradient Ascent:** Execution disabled by RUN_FINAL_EXPERIMENT.'))

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Verify local start: gradient_ascent / recipient_withdrawal:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8.4 Utility Results
## 8.5 Forgetting Results
## 8.6 Runtime
## 8.7 Findings

Selection uses only training-forget loss and retained-validation protection. Full Retraining remains a post-hoc reference.

In [ ]:
if LOCAL_BASELINE_VERIFICATION_PASSED:
    ga_results = method_results('gradient_ascent')
    display(ga_results.round(4))
else:
    ga_results = pd.DataFrame(columns=['Scenario', *UTILITY_KEYS, 'KS statistic', 'KS p-value', 'Training seconds', 'Speed-up'])
    display(Markdown('No Gradient Ascent result table is shown because local execution is disabled.'))

# 9. Gradient Difference

## 9.1 Method
Each forget row is paired with one deterministically sampled retained-training row. One combined backward pass minimises `L_retain - L_forget`.

## 9.2 Configuration
The final MLP rule is retained: AdamW `1e-5`, zero weight decay, λ=1, clipping 1, and five fixed epochs. Epoch 5 is primary; there is no validation, test, Full Retraining, or KS checkpoint selection. Batch 8 with accumulation 4 is the Qwen/MPS adaptation.

## 9.3 Training

In [ ]:
class PairedDataset(Dataset):
    def __init__(self, forget, retain, indices): self.forget, self.retain, self.indices = forget.reset_index(drop=True), retain.reset_index(drop=True), indices
    def __len__(self): return len(self.forget)
    def __getitem__(self, i): return self.forget.iloc[i].to_dict(), self.retain.iloc[int(self.indices[i])].to_dict()
def paired_collate(rows, tokenizer):
    forget, retain = zip(*rows); return collate_text(forget, tokenizer), collate_text(retain, tokenizer)
def train_gd(model, tokenizer, parts, scenario):
    cfg = METHOD_CONFIGS['gradient_difference']; parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=cfg['learning_rate'], weight_decay=0)
    history, started = [], time.perf_counter()
    for epoch in range(1, cfg['epochs'] + 1):
        rng = np.random.default_rng(SEED + 20_000 * (SCENARIOS.index(scenario) + 1) + epoch)
        indices = rng.integers(0, len(parts['retained_train']), size=len(parts['training_forget']))
        dataset = PairedDataset(parts['training_forget'], parts['retained_train'], indices)
        loader = DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True, num_workers=0,
            generator=torch.Generator().manual_seed(SEED + epoch), collate_fn=lambda rows: paired_collate(rows, tokenizer))
        model.train(); optimizer.zero_grad(set_to_none=True); totals = {'forget': 0.0, 'retain': 0.0, 'combined': 0.0}
        for step, (forget_batch, retain_batch) in enumerate(tqdm(loader, desc=f'GD {scenario} epoch {epoch}'), 1):
            fy, ry = forget_batch['labels'].to(DEVICE), retain_batch['labels'].to(DEVICE)
            forget_loss = batch_ce(final_logits(model, forget_batch), fy)
            retain_loss = batch_ce(final_logits(model, retain_batch), ry)
            objective = retain_loss - cfg['forget_weight_lambda'] * forget_loss
            (objective / cfg['gradient_accumulation']).backward()
            if step % cfg['gradient_accumulation'] == 0 or step == len(loader):
                torch.nn.utils.clip_grad_norm_(parameters, cfg['gradient_clip_norm']); optimizer.step(); optimizer.zero_grad(set_to_none=True)
            n = len(fy); totals['forget'] += float(forget_loss.detach()) * n; totals['retain'] += float(retain_loss.detach()) * n; totals['combined'] += float(objective.detach()) * n
        history.append({'scenario': scenario, 'epoch': epoch, 'mean_forget_bce': totals['forget'] / len(dataset),
          'mean_retain_bce': totals['retain'] / len(dataset), 'mean_combined_objective': totals['combined'] / len(dataset),
          'forget_examples': len(dataset), 'sampled_retain_examples': len(dataset)})
    return model, history, {'selected_epoch': cfg['epochs'], 'epochs_executed': cfg['epochs'],
      'training_seconds': time.perf_counter() - started}

In [ ]:
if RUN_FINAL_EXPERIMENT and LOCAL_BASELINE_VERIFICATION_PASSED:
    for scenario in SCENARIOS:
        APPROXIMATE_RUN_REQUESTS.append(('gradient_difference', scenario))
        action = prepare_run('gradient_difference', scenario)
        if action == 'skip': print('GD skip:', scenario); continue
        APPROXIMATE_EXECUTION_ATTEMPTS.append(('gradient_difference', scenario))
        APPROXIMATE_UNLEARNING_EXECUTED = True
        model, tokenizer = fresh_local_baseline('gradient_difference', scenario)
        try:
            model, history, summary = train_gd(model, tokenizer, scenario_sets[scenario], scenario)
            summary['started_from_verified_local_baseline'] = True
            summary['local_baseline_state_sha256'] = LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256']
            evaluate_and_save('gradient_difference', scenario, model, tokenizer, history, summary)
        finally: del model, tokenizer; clear_device_cache()
else:
    if RUN_FINAL_EXPERIMENT: raise RuntimeError('Gradient Difference requires a reproducible local baseline.')
    display(Markdown('**Gradient Difference:** Execution disabled by RUN_FINAL_EXPERIMENT.'))

## 9.4 Utility Results
## 9.5 Forgetting Results
## 9.6 Runtime
## 9.7 Findings

The fixed trajectory prevents test or forgetting evidence from becoming a model-selection oracle.

In [ ]:
if LOCAL_BASELINE_VERIFICATION_PASSED:
    gd_results = method_results('gradient_difference')
    display(gd_results.round(4))
else:
    gd_results = pd.DataFrame(columns=['Scenario', *UTILITY_KEYS, 'KS statistic', 'KS p-value', 'Training seconds', 'Speed-up'])
    display(Markdown('No Gradient Difference result table is shown because local execution is disabled.'))

# 10. Overall Comparison

## 10.1 Retained Utility
## 10.2 Forgetting
## 10.3 Computational Efficiency
## 10.4 Comparison Across Deletion Scenarios

Historical Original utility comes from frozen saved probabilities and Full Retraining is never rerun. RSFT, GA, and GD are reported when the separate local experimental baseline passes strict reproducibility verification. The historical A100 reproduction outcome never suppresses these tables.

In [ ]:
METHOD_LABELS = {'retain_set_fine_tuning': 'Retain-Set Fine-Tuning',
 'gradient_ascent': 'Gradient Ascent', 'gradient_difference': 'Gradient Difference'}
utility_rows, forgetting_rows, runtime_rows = [], [], []
for scenario in SCENARIOS:
    utility_rows.append({'Scenario': SCENARIO_LABELS[scenario], 'Model': 'Historical Original Qwen (A100)',
        **scenario_original[scenario]})
    utility_rows.append({'Scenario': SCENARIO_LABELS[scenario], 'Model': 'Local Experimental Baseline',
        **LOCAL_SCENARIO_BASELINE_METRICS[scenario]})
    utility_rows.append({'Scenario': SCENARIO_LABELS[scenario], 'Model': 'Full Retraining',
        **{k: full_retraining[scenario]['metrics'][k] for k in UTILITY_KEYS}})
    if LOCAL_BASELINE_VERIFICATION_PASSED:
        for method, label in METHOD_LABELS.items():
            p = run_paths(method, scenario)
            if not p['complete'].is_file():
                continue
            complete = json.loads(p['complete'].read_text())
            metrics = pd.read_csv(p['result'] / 'retained_test_metrics.csv').iloc[0]
            utility_rows.append({'Scenario': SCENARIO_LABELS[scenario], 'Model': label, **{k: metrics[k] for k in UTILITY_KEYS}})
            forgetting_rows.append({'Scenario': SCENARIO_LABELS[scenario], 'Method': label,
              'KS statistic': complete['ks_statistic'], 'KS p-value': complete['ks_p_value']})
            runtime_rows.append({'Scenario': SCENARIO_LABELS[scenario], 'Method': label,
              'Approximate seconds': complete['training_seconds'], 'Full Retraining seconds': complete['full_retraining_seconds'],
              'Speed-up': complete['speed_up']})
overall_utility = pd.DataFrame(utility_rows)
overall_forgetting = pd.DataFrame(forgetting_rows, columns=['Scenario', 'Method', 'KS statistic', 'KS p-value'])
overall_runtime = pd.DataFrame(runtime_rows, columns=['Scenario', 'Method', 'Approximate seconds', 'Full Retraining seconds', 'Speed-up'])
display(overall_utility.round(4))
if LOCAL_BASELINE_VERIFICATION_PASSED:
    display(overall_forgetting.round(4)); display(overall_runtime.round(3))
else:
    display(Markdown('Approximate tables are omitted only because local experiment execution is disabled.'))

## 10.5 Main Findings

Interpret retained utility, behavioural similarity, and runtime together but do not collapse them into a model-selection score. Lower KS indicates greater similarity to Full Retraining; runtime is hardware-dependent.

## Final LLM Extension Outcome

The outcome below is generated from the verification state and the genuinely available frozen outputs.

In [ ]:
frozen_inputs_unchanged = frozen_state() == FROZEN_STATE_BEFORE
validation_rows = []
for method, label in METHOD_LABELS.items():
    for scenario in SCENARIOS:
        p = run_paths(method, scenario)
        requested = (method, scenario) in APPROXIMATE_RUN_REQUESTS
        completed = p['complete'].is_file() and all(path.is_file() for path in required_run_files(method, scenario))
        started_verified = utility_available = ks_available = runtime_available = False
        if completed:
            complete = json.loads(p['complete'].read_text())
            configuration = json.loads((p['result'] / 'configuration.json').read_text())
            metrics = pd.read_csv(p['result'] / 'retained_test_metrics.csv')
            forgetting = pd.read_csv(p['result'] / 'forgetting_metrics.csv')
            runtime = json.loads((p['result'] / 'runtime.json').read_text())
            started_verified = bool(
                configuration.get('started_from_verified_local_baseline') is True and
                configuration.get('local_baseline_state_sha256') == LOCAL_BASELINE_CONFIGURATION['trainable_state_sha256'])
            utility_available = bool(len(metrics) == 1 and
                np.isfinite(metrics.loc[0, UTILITY_KEYS].astype(float)).all())
            ks_available = bool(len(forgetting) == 1 and
                np.isfinite(forgetting.loc[0, ['ks_statistic', 'ks_p_value']].astype(float)).all())
            runtime_available = bool(np.isfinite(float(runtime.get('training_seconds', np.nan))))
        validation_rows.append({'Method': label, 'Scenario': SCENARIO_LABELS[scenario],
            'Run requested?': requested, 'Started from verified local baseline?': started_verified,
            'Completed?': completed, 'Utility available?': utility_available,
            'KS available?': ks_available, 'Runtime available?': runtime_available})
run_validation = pd.DataFrame(validation_rows)
display(run_validation)
required_validation_columns = ['Run requested?', 'Started from verified local baseline?', 'Completed?',
    'Utility available?', 'KS available?', 'Runtime available?']
all_15_runs_valid = bool(len(run_validation) == 15 and run_validation[required_validation_columns].all().all())
all_approximate_runs_complete = bool(run_validation['Completed?'].all())

final_component_status = pd.DataFrame([
    {'Component': 'Historical A100 Baseline', 'Status': 'Available'},
    {'Component': 'Historical reproduction on Mac',
     'Status': 'Reproduced' if HISTORICAL_BASELINE_REPRODUCED else 'Failed / not numerically identical'},
    {'Component': 'Local Experimental Baseline',
     'Status': 'Verified' if LOCAL_BASELINE_VERIFICATION_PASSED else 'Failed'},
    *[{'Component': label,
       'Status': 'Run / Complete' if run_validation.loc[run_validation.Method.eq(label), 'Completed?'].all() else 'Incomplete'}
      for label in METHOD_LABELS.values()]
])
display(final_component_status)
if LOCAL_BASELINE_VERIFICATION_PASSED and all_15_runs_valid:
    LLM_APPROX_UNLEARNING_STATUS = 'completed'
elif LOCAL_BASELINE_VERIFICATION_PASSED:
    LLM_APPROX_UNLEARNING_STATUS = 'incomplete'
STATUS_PATH.write_text(json.dumps(current_status_payload(), indent=2), encoding='utf-8')

same_verified_start = bool(run_validation['Started from verified local baseline?'].all())
fifteen_requests = (len(APPROXIMATE_RUN_REQUESTS) == 15 and len(set(APPROXIMATE_RUN_REQUESTS)) == 15)
final_safety_checks = pd.DataFrame([
    {'Check': 'Historical A100 mismatch remains documented and informational',
     'Passed': HISTORICAL_BASELINE_VERIFICATION is not None and not current_status_payload()['historical_check_is_execution_gate']},
    {'Check': 'Historical verification tolerance was not relaxed',
     'Passed': VERIFICATION_ATOL == 1e-5 and VERIFICATION_RTOL == 1e-4},
    {'Check': 'Local experimental baseline state and evidence were frozen',
     'Passed': LOCAL_BASELINE_FROZEN and bool(LOCAL_BASELINE_STATE) and bool(LOCAL_BASELINE_HEAD_STATE)},
    {'Check': 'Independent local baseline reproducibility passed',
     'Passed': LOCAL_BASELINE_VERIFICATION_PASSED},
    {'Check': 'Every approximate run used the same verified local start state',
     'Passed': same_verified_start},
    {'Check': 'All 15 method–scenario runs were requested exactly once', 'Passed': fifteen_requests},
    {'Check': 'All 15 runs have valid utility, KS, p-value, and runtime outputs',
     'Passed': all_15_runs_valid},
    {'Check': 'Frozen historical Original and Full Retraining artefacts are unchanged',
     'Passed': frozen_inputs_unchanged},
    {'Check': 'Post-unlearning differences were retained as results, not equality failures',
     'Passed': all_approximate_runs_complete},
    {'Check': 'No result was fabricated',
     'Passed': all_15_runs_valid and len(overall_forgetting) == 15 and len(overall_runtime) == 15},
    {'Check': 'Machine-readable status file saved', 'Passed': STATUS_PATH.is_file()},
])
display(final_safety_checks)
if not final_safety_checks.Passed.all():
    raise RuntimeError('Final local-baseline or approximate-unlearning validation failed.')
display(Markdown(
    '**Final finding:** The historical A100 reproduction check remains visible but informational. '
    'The local experimental baseline reproduced within the unchanged strict tolerance, and all 15 '
    'approximate-unlearning runs independently started from that same frozen local state. Differences '
    'after unlearning are reported as experimental results rather than treated as verification failures.'))
print('Notebook complete. Frozen historical inputs are unchanged. Status:', LLM_APPROX_UNLEARNING_STATUS)